<table style="width: 100%; border-collapse: collapse; border: none; background: #eef2ff; border-left: 6px solid #4f46e5; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #1e1b4b; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Agentes Basados en Conocimiento: Lógica Proposicional 🧠🔗
      </h1>
      <p style="margin: 6px 0 0 0; color: #4338ca; font-size: 1.1em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial — Taller Práctico Evaluativo
      </p>
      <p style="margin: 4px 0 0 0; color: #3730a3; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #4f46e5; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 05 • Taller Hands-On
      </span><br>
      <span style="color: #1e1b4b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #4338ca; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

---
## 🎯 Objetivos del Taller

En este taller trabajarás los contenidos del **Módulo 05: Agentes Basados en Conocimiento**:

1. **Model-checking:** construir una base de conocimiento en lógica proposicional y resolver un acertijo lógico nuevo.
2. **Forma Normal Conjuntiva (CNF):** convertir manualmente 3 fórmulas a CNF y verificar la conversión en código.
3. **Diseño propio:** crear tu propio acertijo lógico de al menos 3 pistas y resolverlo con el motor de inferencia.

> ⚠️ **Instrucciones de entrega:** completa cada celda marcada con `# TODO:`. No se aceptan notebooks con celdas sin ejecutar o con errores. Antes de entregar, usa *Restart Kernel and Run All Cells* para verificar que todo el notebook corre de principio a fin.

---
### 📦 Mini-Librería de Lógica Proposicional (ya provista, no la modifiques)

Esta es una copia funcional de una pequeña librería de lógica proposicional: símbolos, conectores (`Not`, `And`, `Or`, `Implication`, `Biconditional`) y un motor de **model-checking** (`model_check`) que verifica si una base de conocimiento implica lógicamente una consulta, probando por fuerza bruta todos los modelos posibles.

In [ ]:
import itertools


class Symbol:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return self.name
    def __hash__(self):
        return hash(("symbol", self.name))
    def __eq__(self, other):
        return isinstance(other, Symbol) and self.name == other.name
    def evaluate(self, model):
        try:
            return model[self.name]
        except KeyError:
            raise Exception(f"La variable {self.name} no está definida en el modelo")
    def symbols(self):
        return {self.name}


class Not:
    def __init__(self, operand):
        self.operand = operand
    def __repr__(self):
        return f"¬({self.operand})"
    def evaluate(self, model):
        return not self.operand.evaluate(model)
    def symbols(self):
        return self.operand.symbols()


class And:
    def __init__(self, *conjuncts):
        self.conjuncts = list(conjuncts)
    def __repr__(self):
        return "(" + " ∧ ".join(str(c) for c in self.conjuncts) + ")"
    def evaluate(self, model):
        return all(c.evaluate(model) for c in self.conjuncts)
    def symbols(self):
        return set.union(*[c.symbols() for c in self.conjuncts]) if self.conjuncts else set()


class Or:
    def __init__(self, *disjuncts):
        self.disjuncts = list(disjuncts)
    def __repr__(self):
        return "(" + " ∨ ".join(str(d) for d in self.disjuncts) + ")"
    def evaluate(self, model):
        return any(d.evaluate(model) for d in self.disjuncts)
    def symbols(self):
        return set.union(*[d.symbols() for d in self.disjuncts]) if self.disjuncts else set()


class Implication:
    def __init__(self, antecedent, consequent):
        self.antecedent = antecedent
        self.consequent = consequent
    def __repr__(self):
        return f"({self.antecedent} → {self.consequent})"
    def evaluate(self, model):
        return (not self.antecedent.evaluate(model)) or self.consequent.evaluate(model)
    def symbols(self):
        return self.antecedent.symbols() | self.consequent.symbols()


class Biconditional:
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self):
        return f"({self.left} ↔ {self.right})"
    def evaluate(self, model):
        return self.left.evaluate(model) == self.right.evaluate(model)
    def symbols(self):
        return self.left.symbols() | self.right.symbols()


def model_check(knowledge, query):
    """Verifica si `knowledge` implica lógicamente `query` (entailment),
    probando todos los modelos posibles (fuerza bruta) sobre los símbolos
    involucrados en ambas fórmulas."""
    simbolos = knowledge.symbols() | query.symbols()

    def check_all(pendientes, model):
        if not pendientes:
            if knowledge.evaluate(model):
                return query.evaluate(model)
            return True
        resto = pendientes.copy()
        s = resto.pop()
        modelo_true = model.copy(); modelo_true[s] = True
        modelo_false = model.copy(); modelo_false[s] = False
        return check_all(resto, modelo_true) and check_all(resto, modelo_false)

    return check_all(simbolos, {})


def formulas_equivalentes(f1, f2):
    """Devuelve True si f1 y f2 son lógicamente equivalentes (mismo valor de
    verdad en TODOS los modelos posibles). Útil para verificar una conversión a CNF."""
    simbolos = list(f1.symbols() | f2.symbols())
    for valores in itertools.product([True, False], repeat=len(simbolos)):
        modelo = dict(zip(simbolos, valores))
        if f1.evaluate(modelo) != f2.evaluate(modelo):
            return False
    return True


print("Mini-librería de lógica proposicional cargada correctamente ✅")

---
### 📌 Reto 1: Acertijo de Caballeros y Escuderos

En la isla de los Caballeros y Escuderos, cada habitante es **Caballero** (siempre dice la verdad) o **Escudero** (siempre miente), nunca ambos. Tres habitantes hacen las siguientes declaraciones:

- **Ana dice:** "Beto es Escudero."
- **Beto dice:** "Ana y Cami son del mismo tipo."
- **Cami dice:** "Yo soy Caballero o Ana es Escudero."

**Ejercicio 1.1:** Construye la base de conocimiento `conocimiento` como un `And(...)` que incluya:
1. Cada persona es Caballero o Escudero, pero no ambos (usa `Or` y `Not`/`And` para el "o exclusivo").
2. Cada declaración es verdadera **si y solo si** quien la dice es Caballero (usa `Biconditional`).

Luego usa `model_check` para determinar, para cada persona, si es Caballero.

In [ ]:
Ana_Caballero = Symbol("Ana_Caballero")
Beto_Caballero = Symbol("Beto_Caballero")
Cami_Caballero = Symbol("Cami_Caballero")

# TODO: construye `conocimiento` como un And(...) que incluya:
# 1) Para cada persona: Or(Persona_Caballero, Not(Persona_Caballero)) no basta,
#    en este modelo "no Caballero" ya implica "Escudero", así que basta con
#    modelar directamente las 3 implicaciones de las declaraciones (paso 2).
# 2) La afirmación de Ana ("Beto es Escudero", es decir Not(Beto_Caballero)) es
#    verdadera si y solo si Ana es Caballero:
#    Biconditional(Not(Beto_Caballero), Ana_Caballero)
# 3) La afirmación de Beto ("Ana y Cami son del mismo tipo") es verdadera si y
#    solo si Beto es Caballero:
#    Biconditional(Biconditional(Ana_Caballero, Cami_Caballero), Beto_Caballero)
# 4) La afirmación de Cami ("Yo soy Caballero o Ana es Escudero") es verdadera
#    si y solo si Cami es Caballero:
#    Biconditional(Or(Cami_Caballero, Not(Ana_Caballero)), Cami_Caballero)
# Combina las 3 con And(...)

conocimiento = None  # TODO

for nombre, simbolo in [("Ana", Ana_Caballero), ("Beto", Beto_Caballero), ("Cami", Cami_Caballero)]:
    es_caballero = None  # TODO: usa model_check(conocimiento, simbolo)
    print(f"{nombre} es Caballero:", es_caballero)

---
### 📌 Reto 2: Conversión Manual a Forma Normal Conjuntiva (CNF)

Una fórmula está en **Forma Normal Conjuntiva (CNF)** cuando es una conjunción (`And`) de cláusulas, donde cada cláusula es una disyunción (`Or`) de literales (símbolos o su negación), sin usar `Implication` ni `Biconditional`.

**Ejercicio 2.1 (a mano, en esta celda):** convierte cada una de las siguientes 3 fórmulas a CNF, mostrando el paso a paso (elimina bicondicionales, elimina implicaciones, aplica De Morgan, aplica distributividad):

1. `F1 = P → (Q ∨ R)`
2. `F2 = ¬(P ∧ Q)`
3. `F3 = (P ↔ Q)`

*(Escribe aquí tu desarrollo paso a paso en LaTeX o texto plano antes de pasar a la celda de código.)*

**Ejercicio 2.2:** en la celda de código, construye la versión CNF de cada fórmula (usando solo `Symbol`, `Not`, `And`, `Or`) y verifica con `formulas_equivalentes` que sea lógicamente equivalente a la fórmula original.

In [ ]:
P = Symbol("P")
Q = Symbol("Q")
R = Symbol("R")

# Fórmulas originales (ya construidas, no las modifiques)
f1_original = Implication(P, Or(Q, R))   # P → (Q ∨ R)
f2_original = Not(And(P, Q))              # ¬(P ∧ Q)
f3_original = Biconditional(P, Q)         # (P ↔ Q)

# TODO: construye la versión en CNF de cada fórmula (solo And, Or, Not),
# de acuerdo con la conversión manual que hiciste en la celda markdown anterior.
f1_cnf = None  # TODO
f2_cnf = None  # TODO
f3_cnf = None  # TODO

print("F1 original ≡ F1 CNF:", formulas_equivalentes(f1_original, f1_cnf))
print("F2 original ≡ F2 CNF:", formulas_equivalentes(f2_original, f2_cnf))
print("F3 original ≡ F3 CNF:", formulas_equivalentes(f3_original, f3_cnf))

---
### 📌 Reto 3: Diseña y Resuelve tu Propio Acertijo Lógico

**Ejercicio 3.1:** Inventa tu propio acertijo lógico con **al menos 3 pistas** (puede ser del estilo Caballeros/Escuderos, sospechosos de un crimen, ubicación de objetos, etc.). Define los símbolos necesarios, construye tu base de conocimiento `mi_conocimiento` a partir de tus 3 (o más) pistas, y usa `model_check` para responder al menos una pregunta de interés sobre tu acertijo.

In [ ]:
# TODO 1: describe tu acertijo en un comentario (enunciado + al menos 3 pistas)

# TODO 2: define los símbolos que necesites, ej:
# Sospechoso_A = Symbol("Sospechoso_A")

# TODO 3: construye mi_conocimiento como un And(...) que represente tus pistas
mi_conocimiento = None  # TODO

# TODO 4: usa model_check para responder al menos una pregunta de tu acertijo
respuesta = None  # TODO
print("Respuesta a mi acertijo:", respuesta)

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial</i><br>
    Taller Hands-On — Módulo 05: Agentes Basados en Conocimiento
  </p>
</div>